## Cell 1 — Imports

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

from xgboost import XGBClassifier
import joblib

warnings.filterwarnings("ignore")
print("Libraries loaded successfully.")

## Cell 2 — Project Paths

In [ ]:
BASE_DIR       = Path("..")
RAW_DIR        = BASE_DIR / "data" / "raw"
PROCESSED_DIR  = BASE_DIR / "data" / "processed"
MODELS_DIR     = BASE_DIR / "models"
REPORTS_DIR    = BASE_DIR / "reports"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

DATASET_PATH = RAW_DIR / "manganese_prospectivity_MP_MH_1km.csv"

if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATASET_PATH.resolve()}")

print("Dataset:", DATASET_PATH.resolve())
print(f"File size: {DATASET_PATH.stat().st_size / (1024**2):.1f} MB")

## Cell 3 — Load Dataset

In [ ]:
df = pd.read_csv(DATASET_PATH)
print("Shape:", df.shape)
print("\nFirst 3 rows:")
display(df.head(3))

## Cell 4 — Basic EDA

In [ ]:
print("Label distribution:")
print(df["label"].value_counts())
print("\n  -1 = unsurveyed (PREDICT set)")
print("   0 = non-prospective")
print("   1 = prospective (manganese deposit likely)")

print("\nSplit distribution:")
print(df["split"].value_counts())

print("\nMissing values (top 10):")
print(df.isnull().sum().sort_values(ascending=False).head(10))

## Cell 5 — Separate Labelled vs Predict Set

In [ ]:
# Only TRAIN / TEST rows have real labels (0 or 1)
labelled_df = df[df["label"] != -1].copy()
predict_df  = df[df["label"] == -1].copy()

print("Labelled rows (TRAIN + TEST):", len(labelled_df))
print("Predict rows  (unsurveyed)   :", len(predict_df))

print("\nLabel distribution in labelled set:")
print(labelled_df["label"].value_counts())

print("\nClass balance (%):")
print((labelled_df["label"].value_counts(normalize=True) * 100).round(2))

## Cell 6 — Define Features

In [ ]:
# Columns to exclude from features
EXCLUDE = [
    "cell_id", "label", "split",
    "label_source", "label_confidence", "label_year", "label_method",
    "spatial_block_id", "tehsil", "district"   # high-cardinality IDs
]

ALL_FEATURES = [c for c in df.columns if c not in EXCLUDE]

numeric_features     = [c for c in ALL_FEATURES
                         if labelled_df[c].dtype in ["float64","int64","float32","int32"]]
categorical_features = [c for c in ALL_FEATURES
                         if labelled_df[c].dtype == "object"]

print("Total features  :", len(ALL_FEATURES))
print("Numeric         :", len(numeric_features))
print("Categorical     :", len(categorical_features))
print("  →", categorical_features)

## Cell 7 — Train / Test Split  
*(Using the pre-existing `split` column)*

In [ ]:
train_df = labelled_df[labelled_df["split"] == "TRAIN"].copy()
test_df  = labelled_df[labelled_df["split"] == "TEST"].copy()

# Fallback: if split column not used, do 80/20 chronological
if len(train_df) == 0:
    print("WARNING: No TRAIN split found — falling back to 80/20.")
    split_idx = int(len(labelled_df) * 0.8)
    train_df  = labelled_df.iloc[:split_idx].copy()
    test_df   = labelled_df.iloc[split_idx:].copy()

print(f"Train rows : {len(train_df)}")
print(f"Test rows  : {len(test_df)}")
print(f"Train label dist:\n{train_df['label'].value_counts()}")

## Cell 8 — Prepare X / y

In [ ]:
TARGET = "label"

X_train = train_df[ALL_FEATURES].copy()
X_test  = test_df[ALL_FEATURES].copy()
X_pred  = predict_df[ALL_FEATURES].copy()

y_train = train_df[TARGET]
y_test  = test_df[TARGET]

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("X_pred :", X_pred.shape)

## Cell 9 — Preprocessing Pipeline

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("numeric",     numeric_pipeline,     numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print("Preprocessor built.")

## Cell 10 — Compute Class Weight for Imbalance

In [ ]:
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
scale_pos_weight = n_neg / n_pos

print(f"Negative (non-prospective) : {n_neg}")
print(f"Positive (prospective)     : {n_pos}")
print(f"scale_pos_weight           : {scale_pos_weight:.2f}")

## Cell 11 — Train XGBoost Prospectivity Classifier

In [ ]:
model = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="auc",
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

print("Training prospectivity classifier...")
pipeline.fit(X_train, y_train)
print("Training complete.")

## Cell 12 — Evaluate on Test Set

In [ ]:
y_pred      = pipeline.predict(X_test)
y_prob      = pipeline.predict_proba(X_test)[:, 1]

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall    = recall_score(y_test, y_pred, zero_division=0)
f1        = f1_score(y_test, y_pred, zero_division=0)
roc_auc   = roc_auc_score(y_test, y_prob)

print("PROSPECTIVITY MODEL — TEST RESULTS")
print("=" * 38)
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"ROC-AUC   : {roc_auc:.4f}")
print()
print(classification_report(y_test, y_pred,
      target_names=["Non-Prospective", "Prospective"]))

## Cell 13 — Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=["Non-Prospective", "Prospective"])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Prospectivity Classifier — Test Set Confusion Matrix")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "prospectivity_confusion_matrix.png", dpi=150)
plt.show()
print("Saved confusion matrix.")

## Cell 14 — Feature Importance (Top 20)

In [ ]:
fitted_preprocessor = pipeline.named_steps["preprocessor"]
fitted_model        = pipeline.named_steps["model"]

feature_names = fitted_preprocessor.get_feature_names_out()
importances   = fitted_model.feature_importances_

fi_df = (pd.DataFrame({"feature": feature_names, "importance": importances})
           .sort_values("importance", ascending=False)
           .head(20))

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(fi_df["feature"][::-1], fi_df["importance"][::-1], color="#4A90D9")
ax.set_xlabel("XGBoost Feature Importance (gain)")
ax.set_title("Top 20 Features — Manganese Prospectivity Model")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "prospectivity_feature_importance.png", dpi=150)
plt.show()
print("Saved feature importance plot.")

## Cell 15 — Predict on Unsurveyed Cells

In [ ]:
predict_proba = pipeline.predict_proba(X_pred)[:, 1]
predict_label = pipeline.predict(X_pred)

results_df = predict_df[["cell_id", "latitude", "longitude", "state"]].copy()
results_df["prospectivity_prob"]  = predict_proba.round(4)
results_df["predicted_label"]     = predict_label
results_df["prospectivity_level"] = pd.cut(
    predict_proba,
    bins=[0, 0.3, 0.6, 0.8, 1.0],
    labels=["LOW", "MEDIUM", "HIGH", "VERY HIGH"]
)

results_df = results_df.sort_values("prospectivity_prob", ascending=False)

print(f"Total unsurveyed cells predicted: {len(results_df)}")
print("\nProspectivity level distribution:")
print(results_df["prospectivity_level"].value_counts())
print("\nTop 10 most prospective cells:")
display(results_df.head(10))

## Cell 16 — Save Predictions CSV

In [ ]:
out_path = PROCESSED_DIR / "prospectivity_predictions.csv"
results_df.to_csv(out_path, index=False)
print(f"Predictions saved to: {out_path.resolve()}")
print(f"Total rows: {len(results_df)}")

## Cell 17 — Save Model

In [ ]:
joblib.dump(pipeline, MODELS_DIR / "prospectivity_classifier.joblib")
print("Model saved:", (MODELS_DIR / 'prospectivity_classifier.joblib').resolve())

## Cell 18 — Save Metrics

In [ ]:
metrics = {
    "model"     : "XGBClassifier",
    "dataset"   : DATASET_PATH.name,
    "train_rows": int(len(train_df)),
    "test_rows" : int(len(test_df)),
    "accuracy"  : round(float(accuracy),  4),
    "precision" : round(float(precision), 4),
    "recall"    : round(float(recall),    4),
    "f1"        : round(float(f1),        4),
    "roc_auc"   : round(float(roc_auc),   4),
}

with open(REPORTS_DIR / "prospectivity_metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

print("Metrics saved.")
print(json.dumps(metrics, indent=2))

## Cell 19 — Summary

In [ ]:
print("=" * 50)
print("MANGANESE PROSPECTIVITY MODEL — COMPLETE")
print("=" * 50)
print(f"Dataset        : {DATASET_PATH.name}")
print(f"Total records  : {len(df):,}")
print(f"Training rows  : {len(train_df):,}")
print(f"Test rows      : {len(test_df):,}")
print(f"Unsurveyed pred: {len(predict_df):,}")
print()
print("TEST METRICS")
print(f"  ROC-AUC   : {roc_auc:.4f}")
print(f"  F1 Score  : {f1:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  Precision : {precision:.4f}")
print()
print("OUTPUTS")
print(f"  Model     : models/prospectivity_classifier.joblib")
print(f"  Pred CSV  : data/processed/prospectivity_predictions.csv")
print(f"  Metrics   : reports/prospectivity_metrics.json")